# `nanoddpm-edm`

>**Goal:** _Fast Diffusion from scratch._

Upgrading `nanoddpm` (MNIST) to CIFAR-10 with a **Mini-UNet, Classifier-Free Guidance, Elucidating Diffusion Models (EDM) training framework, and PCA-FID evaluation.** No black boxes—just raw PyTorch and explicit diffusion math.

---
## 1. Classifier-Free Guidance (CFG)
Standard conditional diffusion requires a separate classifier to steer generation. CFG removes that dependency by training a single model on **paired conditional/unconditional data**.

During training, we randomly drop the class label ~10% of the time, forcing the network to learn:
- `ε_θ(x_t, t, y)` : noise prediction with conditioning
- `ε_θ(x_t, t, ∅)` : noise prediction without conditioning

At inference, we combine them linearly:
$$
\hat{\epsilon} = \epsilon_\theta(x_t, t, \emptyset) + w \cdot \left( \epsilon_\theta(x_t, t, y) - \epsilon_\theta(x_t, t, \emptyset) \right)
$$
Where `w` (`cfg_scale`) controls the guidance strength:
- `w = 1.0` → standard conditional sampling
- `w = 3.0–7.5` → sharper, class-aligned outputs
- `w > 7.5` → artifacts & oversaturation

**Why it works:** `(ε_cond - ε_uncond)` points toward the class data manifold. Scaling it amplifies the "pull" without external gradients.
<br><br>

---
## 2. DDIM: Deterministic Fast Sampling
DDPM sampling is slow because it adds random noise at every reverse step. DDIM reparameterizes the reverse process as a **deterministic ODE**, allowing us to skip steps safely.

Starting from the DDPM reverse mean:
$$
\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right)
$$

We can predict the original image $x_0$ directly:
$$
x_0 = \frac{x_t - \sqrt{1-\bar{\alpha}_t} \epsilon_\theta}{\sqrt{\bar{\alpha}_t}}
$$

DDIM replaces the stochastic update with:
$$
x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} x_0 + \sqrt{1 - \bar{\alpha}_{t-1} - \sigma_t^2} \epsilon_\theta + \sigma_t z
$$
Setting $\sigma_t = 0$ removes the random noise `z`, making the path deterministic. Because the trajectory is fixed, we can evaluate the network at a subset of steps (e.g., `t = 500, 250, 100, 50, 0`) and interpolate cleanly.

**Result:** 10–50 steps instead of 500–1000. Same weights, same training loop, just a swapped sampler.
<br><br>

---
## 3. EDM: Optimized Schedule & ODE Sampling (`Euler`/`Heun`)
EDM (Elucidating Diffusion Models) replaces heuristic noise schedules with a **continuous, mathematically grounded framework**. Instead of discrete timesteps, diffusion is treated as an ordinary differential equation (ODE) in noise space.

- **Log-Normal Noise Sampling:** Draw noise levels from `σ ~ exp(𝒩(P_mean, P_std²))`. This concentrates training on mid-noise regimes where gradients are most informative.
- **I/O Preconditioning:** Wrap the network with analytically derived scalars so it always sees normalized inputs/outputs, stabilizing gradients across the entire trajectory:
  $$
  c_{\text{in}} = \frac{1}{\sqrt{\sigma^2+1}}, \quad c_{\text{noise}} = \frac{\log\sigma}{4}
  $$
  $$
  D(x, \sigma) = c_{\text{skip}}(\sigma)\,x + c_{\text{out}}(\sigma)\,F\!\left(x \cdot c_{\text{in}}, c_{\text{noise}}\right)
  $$
  Where $c_{\text{skip}} = \frac{1}{\sigma^2+1}$ and $c_{\text{out}} = \frac{\sigma}{\sqrt{\sigma^2+1}}$.
- **Deterministic ODE Sampler:** The reverse process becomes a first-order ODE: $\frac{dx}{d\sigma} = \frac{x - D(x,\sigma)}{\sigma}$. Discretizing gives two solver options:

  **Euler (1st-order, fast):**
  $$
  x_{\text{next}} = x + (\sigma_{\text{next}} - \sigma) \cdot \frac{x - D(x,\sigma)}{\sigma}
  $$

  **Heun (2nd-order, quality):** Predictor-corrector refinement:
  $$
  \begin{aligned}
  x_{\text{pred}} &= x + (\sigma_{\text{next}} - \sigma) \cdot \frac{x - D(x,\sigma)}{\sigma} \quad \text{(predict)} \\
  x_{\text{next}} &= x + \frac{\sigma_{\text{next}} - \sigma}{2} \cdot \left( \frac{x - D(x,\sigma)}{\sigma} + \frac{x_{\text{pred}} - D(x_{\text{pred}},\sigma_{\text{next}})}{\sigma_{\text{next}}} \right) \quad \text{(correct)}
  \end{aligned}
  $$

**Tradeoff:** Heun requires 2× network calls per step but typically achieves 10–20% lower PCA-FID at the same step count. Use `--solver heun` for quality-critical runs, `--solver euler` for rapid iteration.

**Result:** No more `alpha_bar` or discrete timestep indexing. Training converges faster, gradients stay stable at higher resolutions, and sampling naturally supports arbitrary step counts with interchangeable solvers.

----

## 4. Sampling Method Comparison: `EDM` vs `DDIM`

| Aspect | DDIM | EDM Sampler |
|--------|------|-------------|
| **Parameterization** | Discrete `t` steps with `α_t`/`β_t` schedule | Continuous noise levels `σ` (sigma) |
| **Math Foundation** | Reparameterizes DDPM reverse mean into a deterministic loop | Solves the diffusion ODE `dx/dσ = (x - D(x,σ))/σ` via Euler/Heun |
| **Step Skipping** | Requires re-deriving reverse equations for arbitrary step counts | Naturally supports any number of steps; just change the `σ` grid |
| **Stability** | Can drift or saturate at high resolutions without careful clipping | Preconditioning (`c_in`, `c_out`, `c_skip`) keeps gradients stable across all `σ` |

<br><br>

---
## 5. PCA-FID: Lightweight Quality Tracking
Standard FID uses Inception-V3 to extract 2048-d features—heavy, opaque, and overkill for 32×32 images. PCA-FID replaces the black-box extractor with **unsupervised dimensionality reduction**:

1. Flatten images to `[B, 3072]`
2. Compute top-32 principal components across real + generated batches
3. Project both sets into the 32D subspace
4. Run the standard FID formula on the projected means/covariances:

$$
\text{PCA-FID} = \|\mu_r - \mu_g\|^2 + \text{Tr}\!\left(\Sigma_r + \Sigma_g - 2\sqrt{\Sigma_r \Sigma_g}\right)
$$

**Why it's better for educational builds:**
- No external weights or network downloads
- Runs in <0.1s on CPU
- Captures global structure & color distribution
- Fully transparent: principal components are inspectable
<br><br>

---

## Imports & Config

In [ ]:
import torch, torch.nn as nn, torch.optim as optim, torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt, numpy as np, math, json, torch.nn.functional as F
from tqdm import trange, tqdm
import ipywidgets as widgets
import time

import base64
from IPython.display import Image, display

%matplotlib inline
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === CONFIG (EDM-Aligned) ===
EPOCHS = 20          # Training epochs
BATCH_SIZE = 32      # Samples per batch
RESIZE = 32          # Image resolution (16 for speed, 32 for quality)
CFG_SCALE = 4.0      # Classifier-Free Guidance strength

# EDM Noise Schedule (replaces T_STEPS)
P_MEAN = -1.2        # Mean of log-normal noise distribution
P_STD = 1.2          # Std dev of log-normal noise distribution
SIGMA_MIN = 0.002    # Minimum noise level (near clean)
SIGMA_MAX = 80.0     # Maximum noise level (pure noise)
SAMPLE_STEPS = 20    # Discretization steps for EDM Euler sampler

print(f"▶ Device: {device} | Res: {RESIZE} | CFG: {CFG_SCALE} | Sample Steps: {SAMPLE_STEPS}")

## EDM: Optimized Schedule & ODE Sampling

In [ ]:
class EDMWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, sigma, labels=None):
        c_in = 1 / torch.sqrt(sigma**2 + 1)
        c_noise = torch.log(sigma) / 4
        out = self.model(x * c_in[:,None,None,None], c_noise, labels)
        c_skip, c_out = 1/(sigma**2+1), sigma/torch.sqrt(sigma**2+1)
        return c_skip[:,None,None,None]*x + c_out[:,None,None,None]*out

    def sample_sigmas(self, n, device, P_mean=-1.2, P_std=1.2):
        return torch.exp(P_mean + P_std * torch.randn(n, device=device))


## Dataset Loading & Preparation

In [ ]:
transform = T.Compose([T.Resize(RESIZE), T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])
dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
real_batch, real_labels = next(iter(torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False)))
real_batch = real_batch.to(device)
print(f"▶ CIFAR-10 loaded. Shape: {dataset[0][0].shape}")

## Mini-UNet Model

In [ ]:
_res_block = """
graph TD
    A[Input x] --> B[Conv2D 3x3]
    B --> C[GroupNorm]
    C --> D[SiLU]

    E[Time t] --> F[Sinusoidal Emb]
    F --> G[MLP: SiLU + Linear]
    G --> H((+))

    I[Class y] --> J[Embedding]
    J --> K[Dropout 0.1]
    K --> H

    H --> L[Add to Feature Map]
    D --> L

    L --> M[Conv2D 3x3]
    M --> N[GroupNorm]
    N --> O[SiLU]

    A --> P[Skip: Conv 1x1 or Identity]
    P --> Q((+))
    O --> Q
    Q --> R[Output]
"""

_mini_UNET = """
graph TD
    A[Input: 3xHxW] --> D1[ResBlock: 3→ch]
    D1 -->|Skip d1| S1[(d1)]

    D1 --> P1[AvgPool 2x2]
    P1 --> D2[ResBlock: ch→ch*2]
    D2 -->|Skip d2| S2[(d2)]

    D2 --> P2[AvgPool 2x2]
    P2 --> M[ResBlock: ch*2→ch*2]

    M --> U1[Int: 2x Nearest]
    S2 --> C1[Concat]
    U1 --> C1
    C1 --> U1B[ResBlock: ch*3→ch]

    U1B --> U2[Int: 2x Nearest]
    S1 --> C2[Concat]
    U2 --> C2
    C2 --> U2B[ResBlock: ch*2→3]

    U2B --> OUT[Conv2D 1x1]
    OUT --> Z[Output: 3xHxW]
"""

In [ ]:
def render_mermaid(graph_code):
    graphbytes = graph_code.encode("utf-8")
    base64_bytes = base64.b64encode(graphbytes)
    base64_string = base64_bytes.decode("utf-8")
    display(Image(url="https://mermaid.ink/img/" + base64_string))

def render_mermaid_HW(graph_code, width=None, height=None):
    # Fix: Encode and decode using 'utf-8' to handle Unicode characters
    graphbytes = graph_code.encode("utf-8")
    base64_bytes = base64.b64encode(graphbytes)
    base64_string = base64_bytes.decode("utf-8")
    display(Image(url="https://mermaid.ink/img/" + base64_string, width=width, height=height))


In [ ]:
render_mermaid_HW(_res_block, width=600, height=600)

In [ ]:
render_mermaid_HW(_mini_UNET, width=300, height=860)

In [ ]:
def sinusoidal_embedding(t, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, dtype=torch.float32, device=t.device) / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, num_classes=10, dropout=0.1):
        super().__init__()

        # If out_ch is 3, set num_groups to 3. Otherwise, use 8.
        num_groups_gn = min(8, out_ch)

        self.time_dim = time_dim # Store time_dim
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(num_groups_gn, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(num_groups_gn, out_ch)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.class_emb = nn.Embedding(num_classes, time_dim)
        self.class_proj = nn.Linear(time_dim, out_ch)  # Projects class emb to match out_ch
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t, labels=None):
        h = F.silu(self.norm1(self.conv1(x)))
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_dim))

        # Handle CFG: zero embedding for unconditional, projected + dropout for conditional
        if labels is None:
            c_emb = torch.zeros_like(t_emb)
        else:
            c_emb = self.class_proj(self.dropout(self.class_emb(labels)))

        h = h + (t_emb + c_emb)[:, :, None, None]
        return F.silu(self.norm2(self.conv2(h))) + self.skip(x)

class MiniUNet(nn.Module):
    def __init__(self, time_dim=128, ch=32):
        super().__init__()
        self.down1 = ResBlock(3, ch, time_dim)
        self.down2 = ResBlock(ch, ch*2, time_dim)
        self.mid = ResBlock(ch*2, ch*2, time_dim)
        self.up1 = ResBlock(ch*4, ch, time_dim)
        self.up2 = ResBlock(ch*2, 3, time_dim)
        self.out = nn.Conv2d(3, 3, 1)

    def forward(self, x, t, labels=None):
        d1 = self.down1(x, t, labels)
        d2 = self.down2(F.avg_pool2d(d1, 2), t, labels)
        x = self.mid(F.avg_pool2d(d2, 2), t, labels)
        x = self.up1(torch.cat([F.interpolate(x, scale_factor=2), d2], 1), t, labels)
        x = self.up2(torch.cat([F.interpolate(x, scale_factor=2), d1], 1), t, labels)
        return self.out(x)

model = MiniUNet().to(device)
edm_wrapper = EDMWrapper(model).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-3)
print(f"▶ Params: {sum(p.numel() for p in model.parameters()):,}")

## PCA-FID & EDM Sampler

In [ ]:
def pca_fid(real, gen, n_components=32):
    r, g = real.view(real.shape[0], -1).cpu().double(), gen.view(gen.shape[0], -1).cpu().double()
    data = torch.cat([r, g], 0)
    mean = data.mean(0, keepdim=True)
    U, S, V = torch.pca_lowrank(data - mean, q=n_components)
    proj = (data - mean) @ V
    r_p, g_p = proj[:real.shape[0]], proj[real.shape[0]:]
    mu_r, mu_g = r_p.mean(0), g_p.mean(0)
    var_r, var_g = r_p.var(0)+1e-6, g_p.var(0)+1e-6
    return ((mu_r-mu_g)**2).sum().item() + (var_r+var_g-2*torch.sqrt(var_r*var_g)).sum().item()

@torch.no_grad()
def edm_sampler(model, n, labels, cfg=1.0, steps=20, solver='euler'):
    """
    EDM ODE sampler with Euler or Heun solver.
    Args:
        model: EDMWrapper or raw model
        n: number of samples
        labels: class labels tensor
        cfg: classifier-free guidance scale
        steps: number of discretization steps
        solver: 'euler' (1st-order) or 'heun' (2nd-order)
    """
    sigmas = torch.linspace(80.0, 0.002, steps, device=device)
    x = torch.randn(n, 3, RESIZE, RESIZE, device=device) * sigmas[0]

    for i, (s, s_next) in enumerate(zip(sigmas[:-1], sigmas[1:])):
        s_vec = torch.full((n,), s, device=device)

        # CFG-guided prediction
        D_u = model(x, s_vec, None)
        D_c = model(x, s_vec, labels)
        D = D_u + cfg * (D_c - D_u)

        if solver == 'euler':
            x = x + (s_next - s) * (x - D) / s

        elif solver == 'heun':
            # 1. Euler predictor
            x_pred = x + (s_next - s) * (x - D) / s

            # 2. Heun corrector (skip at final step to avoid division by zero)
            if s_next > 1e-5:
                s_next_vec = torch.full((n,), s_next, device=device)
                D_u_pred = model(x_pred, s_next_vec, None)
                D_c_pred = model(x_pred, s_next_vec, labels)
                D_pred = D_u_pred + cfg * (D_c_pred - D_u_pred)

                # Average slope over the step
                x = x + (s_next - s) * ((x - D) / s + (x_pred - D_pred) / s_next) / 2
            else:
                x = x_pred
        else:
            raise ValueError("solver must be 'euler' or 'heun'")

        x = torch.clip(x, -1.0, 1.0)

    return x

## Model Training

In [ ]:
def run_training(epochs, cfg_scale, sample_steps):
    metrics_log = []

    for epoch in trange(1, epochs + 1, desc="Training"):
        model.train()
        epoch_loss, count = 0, 0

        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            sigma = edm_wrapper.sample_sigmas(imgs.shape[0], device)
            x_sigma = imgs + sigma[:, None, None, None] * torch.randn_like(imgs)

            train_labels = labels.clone()
            train_labels[torch.rand_like(labels.float()) < 0.1] = -1

            cond_mask = train_labels != -1
            uncond_mask = ~cond_mask
            pred_x0 = torch.zeros_like(x_sigma)

            if cond_mask.any():
                pred_x0[cond_mask] = edm_wrapper(x_sigma[cond_mask], sigma[cond_mask], train_labels[cond_mask])
            if uncond_mask.any():
                pred_x0[uncond_mask] = edm_wrapper(x_sigma[uncond_mask], sigma[uncond_mask], None)

            c_out = sigma / torch.sqrt(sigma**2 + 1)
            loss_weight = (1.0 / (c_out ** 2)).view(-1, 1, 1, 1)

            optimizer.zero_grad()
            loss = (loss_weight * F.mse_loss(pred_x0, imgs, reduction='none')).mean()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * imgs.shape[0]
            count += imgs.shape[0]

        # === EVALUATION: Fast (Euler) vs Hard (Heun) ===
        model.eval()
        with torch.no_grad():
            samples_fast = edm_sampler(edm_wrapper, n=128, labels=real_labels[:128].to(device), cfg=cfg_scale, steps=sample_steps, solver='euler')
            samples_hard = edm_sampler(edm_wrapper, n=128, labels=real_labels[:128].to(device), cfg=cfg_scale, steps=sample_steps, solver='heun')

            fid_fast = pca_fid(real_batch[:128], samples_fast)
            fid_hard = pca_fid(real_batch[:128], samples_hard)

        metrics_log.append({
            'epoch': epoch,
            'loss': epoch_loss / count,
            'pca_fid_fast': fid_fast,
            'pca_fid_hard': fid_hard
        })
        print(f"  Epoch {epoch:02d} | Loss: {metrics_log[-1]['loss']:.4f} | Fast FID: {fid_fast:.1f} | Hard FID: {fid_hard:.1f}")

    with open('nanoddpm_edm_metrics.json', 'w') as f:
        json.dump(metrics_log, f)
    print("Metrics saved to nanoddpm_edm_metrics.json")
    return metrics_log


In [ ]:
metrics_log = run_training(epochs=EPOCHS, cfg_scale=CFG_SCALE, sample_steps=SAMPLE_STEPS)

## Visualization

In [ ]:
def plot_results(metrics_log, cfg_scale, solver='euler'):
    """Plot training metrics + final EDM-generated samples"""
    suffix = 'fast' if solver == 'euler' else 'hard'
    name = 'Euler (Fast)' if solver == 'euler' else 'Heun (Hard)'

    # Row 1: Metrics
    fig, axs = plt.subplots(1, 2, figsize=(10, 3))
    axs[0].plot([m['epoch'] for m in metrics_log], [m['loss'] for m in metrics_log], marker='o')
    axs[0].set_title('Training Loss')
    axs[0].grid(alpha=0.3)
    axs[1].plot([m['epoch'] for m in metrics_log], [m[f'pca_fid_{suffix}'] for m in metrics_log], marker='s', color='orange')
    axs[1].set_title(f'PCA-FID {name} (↓ better)')
    axs[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Row 2: Final Samples (2 per class)
    print(f"\nFinal samples ({name}, CFG={cfg_scale})")
    model.eval()
    with torch.no_grad():
        all_samples = []
        for cls in range(10):
            labels = torch.full((2,), cls, dtype=torch.long, device=device)
            gen = edm_sampler(edm_wrapper, n=2, labels=labels, cfg=cfg_scale, steps=SAMPLE_STEPS, solver=solver)
            all_samples.append(gen)
        all_samples = torch.cat(all_samples, dim=0)

        grid = torchvision.utils.make_grid(all_samples, nrow=10, normalize=True, value_range=(-1, 1))
        plt.figure(figsize=(12, 2.5))
        plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
        plt.axis('off')
        plt.title(f'Generated Samples (2 per class, {name})')
        plt.tight_layout()
        plt.show()


def plot_results_compare(metrics_log, cfg_scale):
    # === ROW 1: Loss (single) + FID comparison (dual) ===
    fig, axs = plt.subplots(1, 2, figsize=(12, 3))

    # Left: Training Loss (one curve)
    axs[0].plot([m['epoch'] for m in metrics_log], [m['loss'] for m in metrics_log], marker='o')
    axs[0].set_title('Training Loss (shared)')
    axs[0].grid(alpha=0.3)

    # Right: PCA-FID comparison (two curves)
    axs[1].plot([m['epoch'] for m in metrics_log], [m['pca_fid_fast'] for m in metrics_log],
                marker='s', label='Euler (fast)', color='blue')
    axs[1].plot([m['epoch'] for m in metrics_log], [m['pca_fid_hard'] for m in metrics_log],
                marker='^', label='Heun (quality)', color='orange')
    axs[1].set_title('PCA-FID Comparison')
    axs[1].legend()
    axs[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
@torch.no_grad()
def sample_trajectory(n=9, sigmas_to_plot=[80, 40, 20, 10, 2, 0.002], cls_idx=5, cfg_scale=4.0, steps=50, solver='euler'):
    """Visualize denoising from high noise (σ=80) to clean (σ≈0)"""
    model.eval()
    labels = torch.full((n,), cls_idx, device=device)
    start, stop = sigmas_to_plot[0], sigmas_to_plot[-1]
    sigmas = torch.linspace(start, stop, steps, device=device)

    # Initialize noise (scaled by sigma_max)
    x = torch.randn(n, 3, RESIZE, RESIZE, device=device) * sigmas[0]

    # Map to store exactly one image per target sigma
    trajectory_map = {t: None for t in sigmas_to_plot}

    for s, s_next in zip(sigmas[:-1], sigmas[1:]):
        s_vec = torch.full((n,), s, device=device)
        D_u = edm_wrapper(x, s_vec, None)
        D_c = edm_wrapper(x, s_vec, labels)
        D = D_u + cfg_scale * (D_c - D_u)

        if solver == 'euler':
            x = x + (s_next - s) * (x - D) / s
        elif solver == 'heun':
            x_pred = x + (s_next - s) * (x - D) / s
            if s_next > 1e-5:
                s_next_vec = torch.full((n,), s_next, device=device)
                D_u_p = edm_wrapper(x_pred, s_next_vec, None)
                D_c_p = edm_wrapper(x_pred, s_next_vec, labels)
                D_pred = D_u_p + cfg_scale * (D_c_p - D_u_p)
                x = x + (s_next - s) * ((x - D) / s + (x_pred - D_pred) / s_next) / 2
            else:
                x = x_pred

        x = torch.clip(x, -1.0, 1.0)

        # 3. Record image if close to a target sigma
        for target in sigmas_to_plot:
            if abs(s.item() - target) < 2.0:  # Tolerance
                trajectory_map[target] = x.clone().cpu()

    # Filter out missed targets and sort descending (Noise -> Clean)
    valid_sigmas = sorted([k for k, v in trajectory_map.items() if v is not None], reverse=True)
    valid_images = [trajectory_map[k] for k in valid_sigmas]

    if not valid_sigmas:
        print("No images captured. Try increasing 'steps' or 'tolerance'.")
        return

    # Plot
    fig, axes = plt.subplots(1, len(valid_sigmas), figsize=(15, 2))
    if len(valid_sigmas) == 1:
        axes = [axes]  # Handle single subplot case

    for i, (sigma_val, img) in enumerate(zip(valid_sigmas, valid_images)):
        grid = torchvision.utils.make_grid(img, nrow=3, normalize=True, value_range=(-1, 1))
        axes[i].imshow(grid.permute(1, 2, 0).numpy())
        axes[i].set_title(f"σ={sigma_val:.1f} | {solver.upper()}")
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()


#========================================================================================
@torch.no_grad()
def generate(cfg, steps, cls_idx, solver='euler'):
    """
    Generate images with EDM ODE sampling
    Args:
    - cfg (float): CFG scale
    - steps (int): Number of EDM Euler steps
    - cls_idx (int): Class index for conditional generation
    """
    try:
        model.eval()
        n = 9
        cfg, steps, cls_idx = float(cfg), int(steps), int(cls_idx)
        labels = torch.full((n,), cls_idx, dtype=torch.long, device=device)

        # Reuse unified sampler
        x = edm_sampler(edm_wrapper, n=n, labels=labels, cfg=cfg, steps=steps, solver=solver)

        grid = torchvision.utils.make_grid(x, nrow=3, normalize=True, value_range=(-1, 1))
        plt.figure(figsize=(4, 4))
        plt.imshow(grid.cpu().permute(1, 2, 0).numpy())
        plt.title(f'Class {cls_idx} | CFG={cfg} | Steps={steps} | {solver.upper()}')
        plt.axis('off')
        plt.show()
        plt.close()
    except Exception as e:
        print(f"Generation error: {e}")



In [ ]:
plot_results(metrics_log, cfg_scale=CFG_SCALE, solver='euler')  # or 'heun'

In [ ]:
plot_results(metrics_log, cfg_scale=CFG_SCALE, solver='heun')

In [ ]:
plot_results_compare(metrics_log, cfg_scale=CFG_SCALE)

In [ ]:
sample_trajectory(n=9, sigmas_to_plot=[80, 40, 20, 10, 2, 0.002], cls_idx=5, cfg_scale=4.0, solver='euler')

In [ ]:
sample_trajectory(n=9, sigmas_to_plot=[80, 40, 20, 10, 2, 0.002], cls_idx=5, cfg_scale=4.0, solver='heun')

In [ ]:
widgets.interact(
    generate,
    cfg=widgets.FloatSlider(value=4.0, min=1.0, max=7.5, step=0.5, description='CFG Scale'),
    steps=widgets.IntSlider(value=20, min=10, max=50, step=5, description='EDM Steps'),
    cls_idx=widgets.IntSlider(value=0, min=0, max=9, step=1, description='Class (0-9)'),
    solver=widgets.ToggleButtons(options=['euler', 'heun'], description='Solver:')
)

## Compare `Euler` vs `Heun` samplers

In [ ]:
@torch.no_grad()
def compare_samplers(cls_idx=5, cfg=4.0, steps=20):
    """Generate same seed with Euler vs Heun, plot side-by-side"""
    model.eval()
    torch.manual_seed(42)  # Same noise for fair comparison

    labels = torch.full((4,), cls_idx, dtype=torch.long, device=device)

    # Euler samples
    euler = edm_sampler(edm_wrapper, n=4, labels=labels, cfg=cfg, steps=steps, solver='euler')

    # Heun samples (reset seed for identical starting noise)
    torch.manual_seed(42)
    heun = edm_sampler(edm_wrapper, n=4, labels=labels, cfg=cfg, steps=steps, solver='heun')

    # Plot
    fig, axs = plt.subplots(2, 4, figsize=(12, 3))
    for i in range(4):
        axs[0,i].imshow(euler[i].permute(1,2,0).cpu().numpy() * 0.5 + 0.5)
        axs[0,i].axis('off')
        axs[1,i].imshow(heun[i].permute(1,2,0).cpu().numpy() * 0.5 + 0.5)
        axs[1,i].axis('off')
    axs[0,0].set_title('Euler (1st-order)')
    axs[1,0].set_title('Heun (2nd-order)')
    plt.suptitle(f'Class {cls_idx} | CFG={cfg} | Steps={steps}')
    plt.tight_layout()
    plt.show()

    # Quick metric: sample variance (Heun often has slightly higher detail)
    print(f"Euler std: {euler.std().item():.4f} | Heun std: {heun.std().item():.4f}")


In [ ]:
for i in range(1,10):
  compare_samplers(cls_idx=i, cfg=4.0, steps=50)
  print("\n")